# 4. Musculoskeletal Models, Motion Files, and `MuscleAnalysis`

## 4.1. Objectives

**Introduction**

Musculoskeletal models enable us to study neuromuscular coordination, analyze athletic performance, and estimate musculoskeletal loads [[1]](https://mitpress.mit.edu/9780262044202/biomechanics-of-movement/). In [OpenSim](https://opensim.stanford.edu/) [[2]](https://doi.org/10.1109/TBME.2007.901024), a musculoskeletal model consists of rigid body segments connected by joints. Muscles span these joints and generate forces and motion. Once a musculoskeletal model is created, OpenSim enables users to create custom studies, including investigating the effects of musculoskeletal geometry, joint kinematics, and muscle-tendon properties on the forces and joint moments that the muscles can produce. With OpenSim, our goal is to provide a framework that allows the biomechanics community to create, share, and extend a library of models and dynamic simulation tools that can be used to study and quantify human and animal movement. 

**Purpose**

The purpose of this tutorial is to introduce users to the `opensim` package for musculoskeletal modeling by investigating how muscle-tendon lengths and moment arms depend on limb configuration. In this tutorial, you will:

* Become familiar with OpenSim's API.
* Explore a lower-extremity musculoskeletal model.
* Work with motion files.
* Use the `MuscleAnalysis` tool.
* Use OpenSim to analyze an important clinical problem.

**Format**

Each section of the tutorial guides you through certain tools within OpenSim and asks you to answer a few questions. The questions can be answered based on information from OpenSim and basic knowledge of the human musculoskeletal system. As you complete each section of the tutorial, feel free to explore OpenSim and the lower extremity model further on your own.

**Resources**

All input files for this tutorial ship in this folder under `resources/Tutorial 4/`; the setup cell in Section 4.2 copies them into your working directory.

**Prerequisites**

This notebook runs against a locally installed [`opensim`](https://pypi.org/project/opensim/) package (>= 4.6) plus the other packages in `requirements.txt`. Nothing is downloaded - every input file it needs ships in this folder under `resources/`.

## 4.2. Set up OpenSim

Run the cell below to confirm `opensim` imports; it prints the version and build date. If it raises `ModuleNotFoundError`, install the requirements first (`pip install -r requirements.txt`).

In [ ]:
import opensim as osim
print(osim.GetVersionAndDate())

The cell below copies this tutorial's input files from `resources/Tutorial 4/` into an isolated working directory (`_work/Tutorial 4/`) and switches to it, so files can be opened by name and anything the tutorial writes stays out of the repo folder (`_work/` is git-ignored).

In [ ]:
from tutorial_setup import prepare
prepare("Tutorial 4")

## 4.3. Downloading and exploring a lower-extremity model

In this section, you will use a model of the lower extremity [[3]](https://doi.org/10.1109/10.102791), and then load it into OpenSim. The model represents an adult subject with an approximate height of 1.8 m and an approximate mass of 75 kg. The model consists of 13 rigid body segments and includes the lines of action of 92 muscles (43 per leg and 6 at the torso).

First, let's load the model file (`gait2392.osim`) -- bundled in `resources/Tutorial 4/`, already in your working directory.

Now, let's import OpenSim, load the model, and print some metadata.

In [ ]:
import opensim as osim

# Load the model.
gait2392 = osim.Model('gait2392.osim')

# Print metadata.
print("Name of the model:", gait2392.getName())
print("Author:", gait2392.get_credits())
print("Publications:", gait2392.get_publications())
print("Length Unit:", gait2392.get_length_units())
print("Force Unit:", gait2392.get_force_units())
print("Gravity:", gait2392.get_gravity())

Let's explore the coordinates of the model. We know the model has 23 coordinates from the name of the model file.

In [ ]:
# Print the number of coordinates.
print("Num Coordinates:", gait2392.getNumCoordinates())
print()

# For each coordinate, print some information, such as its name or motion type.
for coordinate in gait2392.getCoordinateSet():
    print("  Coordinate Name:", coordinate.getName())
    print("  Coordinate Absolute Path:", coordinate.getAbsolutePathString())
  
    # Motion type is an enumerate (0:Undefined, 1:Rotational, 2:Translational, 3:Coupled).
    motion_type = coordinate.getMotionType()
    motion_type_string = ""
    if motion_type == 0:
        motion_type_string = "Undefined"
    elif motion_type == 1:
        motion_type_string = "Rotational"
    elif motion_type == 2:
        motion_type_string = "Translational"
    elif motion_type == 3:
        motion_type_string = "Coupled"
    print("  Coordinate Motion Type:", motion_type_string)

Let's explore some of the model's joint properties.

In [ ]:
# Print number of joints.
print("Num Joints:", gait2392.getNumJoints())
print()

# For each joint, print some information, such as its name or components.
for joint in gait2392.getJointSet():
    print("Joint Name:", joint.getName())
    print("Joint Absolute Path:", joint.getAbsolutePathString())
    print("Components:")
    for component in joint.getComponentsList():
        print("  Component Name:", component.getName())
        print("  Component Absolute Path:", component.getAbsolutePathString())

And finally, let's explore the muscles in the model. We know the model has 92 muscles from the name of the model file.


In [ ]:
# Print the number of muscles.
print("Num Muscles:", gait2392.getMuscles().getSize())
print()

for muscle in gait2392.getMuscles():
    print("Muscle Name:", muscle.getName())
    print("Muscle Absolute Path:", muscle.getAbsolutePathString())

## 4.4. Working with motion files

In this section, you will load information from a motion file. A motion file (`.mot`) is a tab separated values (tsv) file containing a header with information about the file (e.g., if the values are in degrees, number of columns, etc.), and a set of columns containing how different values in the model evolve over time (e.g., pelvis rotation, pelvis tilt, hip flexion).

Since the motion file already contains the values resulting from a simulation, you will not need to simulate the model.

First, let's load the motion file (`normal_gait.mot`) -- bundled in `resources/Tutorial 4/`, already in your working directory.

Load the file into a `TimeSeriesTable`. Print the column labels to check the file has been processed properly.

In [ ]:
# Use the TableProcessor to read the motion file.
tableTime = osim.TimeSeriesTable('normal_gait.mot')

# Print the column labels.
print(tableTime.getColumnLabels())

In this example, we will plot the knee angle and hip adduction between both legs. Run the following cell to extract this information from the motion file.

In [ ]:
# Get columns we want to analyze, and the time column (independent column).
y_knee_angle_r = tableTime.getDependentColumn('knee_angle_r')
y_knee_angle_l = tableTime.getDependentColumn('knee_angle_l')
y_hip_adduction_r = tableTime.getDependentColumn('hip_adduction_r')
y_hip_adduction_l = tableTime.getDependentColumn('hip_adduction_l')
x_time = tableTime.getIndependentColumn()

We can use Matplotlib to plot the information.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create six subplots, with 2 rows and 3 columns.
fig, axs = plt.subplots(2, 1, figsize=(10, 10))
fig.suptitle('Normal Gait')

# Plot the knee angles on the first subplot.
axs[0].plot(x_time, y_knee_angle_r.to_numpy(), label='knee_angle_r')
axs[0].plot(x_time, y_knee_angle_l.to_numpy(), label='knee_angle_l')
axs[0].set_title('Knee angle')
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Knee angle')
axs[0].grid()
axs[0].legend()

# Plot the hip adductions on the second subplot.
axs[1].plot(x_time, y_hip_adduction_r.to_numpy(), label='hip_adduction_r')
axs[1].plot(x_time, y_hip_adduction_l.to_numpy(), label='hip_adduction_l')
axs[1].set_title('Hip adduction')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Hip Adduction')
axs[1].grid()
axs[1].legend()

# Set the spacing between subplots
plt.subplots_adjust(wspace=0.5, hspace=0.5)

## 4.5. Using `MuscleAnalysis` to assess hamstring length during crouch gait

In this final section of the tutorial, you will use OpenSim to investigate a possible cause of crouch gait, one of the most common walking abnormalities among individuals with cerebral palsy. It is characterized by excessive flexion of the knee during stance phase, which is often accompanied by exaggerated flexion and internal rotation of the hip. One hypothesized cause of crouch gait is short hamstrings, and orthopedic surgeons will sometimes lengthen the hamstrings of such patients in an attempt to improve their posture and gait. However, other causes of excessive knee flexion are possible (e.g., weak ankle plantar flexors), and lengthening the hamstrings can compromise these muscles' force-generation capabilities [[4]](https://doi.org/10.1016/j.gaitpost.2005.03.003). How can a surgeon determine whether a hamstring lengthening procedure is warranted? 

One possible way to judge whether a patient's hamstrings are shorter than "normal" is to develop a musculoskeletal model and compare the length of the hamstrings during the patient's crouch gait cycle to the length of the hamstrings during a normal gait cycle. Suppose that an orthopaedic surgeon has brought you some kinematic data for a patient who walks with a crouch gait. The surgeon is contemplating whether to operate and wants your opinion. 

### 4.5.1. Range of Motion

First, let's load the crouch gait motion file (`crouch_gait.mot`) -- bundled in `resources/Tutorial 4/`, already in your working directory. We already used the normal gait motion file in the previous section.

Load the motion files into `TimeSeriesTable`s. Print the column labels to check the files have been processed properly.

In [ ]:
# Load the normal gait motion file into a TimeSeriesTable.
table_normal_gait = osim.TimeSeriesTable('normal_gait.mot')
# Print labels for each column.
print(table_normal_gait.getColumnLabels())

# Load the crouch gait motion file into a TimeSeriesTable.
table_crouch_gait = osim.TimeSeriesTable('crouch_gait.mot')
# Print labels for each column.
print(table_crouch_gait.getColumnLabels())

Let's obtain the information we want to plot. This time, we obtain two time columns (independent columns), one per each table. Since both have a range from 0 to 1, we can plot them and compare them.

In [ ]:
# Get columns we want to represent.
normal_gait_knee_angle_r = table_normal_gait.getDependentColumn('knee_angle_r')
crouch_gait_knee_angle_r = table_crouch_gait.getDependentColumn('knee_angle_r')

# Get independent columns of each table (time).
normal_gait_time = table_normal_gait.getIndependentColumn()
crouch_gait_time = table_crouch_gait.getIndependentColumn()

Now let's plot both data series to quantitatively compare.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create six subplots, with 2 rows and 3 columns.
fig, axs = plt.subplots(1, 1, figsize=(7, 5))
fig.suptitle('Normal Gait vs. Crouch Gait')

# Plot the knee angles on the first subplot.
axs.plot(normal_gait_time, normal_gait_knee_angle_r.to_numpy(), label='normal_gait_knee_angle_r')
axs.plot(crouch_gait_time, crouch_gait_knee_angle_r.to_numpy(), label='crouch_gait_knee_angle_l')
axs.set_title('Knee angle')
axs.set_xlabel('Time')
axs.set_ylabel('Knee angle')
axs.grid()
axs.legend()

# Set the spacing between subplots
plt.subplots_adjust(wspace=0.5, hspace=0.5)

**Questions: Range of Motion**

**1.** Label the times at which heel strike and toe-off occur, and the stance and swing intervals. 


**2.** What is the range of motion for knee flexion during the stance phase for normal gait?

**3.** How does the knee flexion range of motion for crouch gait compare to that of normal gait?

### 4.5.2. Hamstrings Length

An orthopedic surgeon believes that a patient may benefit from a hamstring lengthening surgery. You are tasked to compare the hamstrings (semitendinosus) length over a patient's crouch gait cycle to the hamstrings length for a "normal" gait cycle. This time, a `MuscleAnalysis` must be performed to obtain the fiber lengths of the muscles.

First, let's load the motion files to obtain the initial and ending time. In both files, the initial and ending times are the same, so we obtain the values from just one of them.

In [ ]:
# Load motion file and extract time.
tableTime = osim.TimeSeriesTable('normal_gait.mot')
x_time = tableTime.getIndependentColumn()

# Get first and last time.
first_time = x_time[0]
last_time = x_time[len(x_time) - 1]

Now, let's create a `MuscleAnalysis` and configure it. This analysis will be applied two times: one for the normal gait and another for the crouch gait.

In [ ]:
# Define a MuscleAnalysis.
muscle_analysis = osim.MuscleAnalysis()

# Set start and end times for the analysis.
muscle_analysis.setStartTime(first_time)
muscle_analysis.setEndTime(last_time)

# Set the muscle of interest (semitendinosus on the right leg).
muscle_list = osim.ArrayStr()
muscle_list.append("semiten_r")
muscle_analysis.setMuscles(muscle_list)

# Configure the analysis.
muscle_analysis.setOn(True)
muscle_analysis.setStepInterval(1)
muscle_analysis.setInDegrees(True)
muscle_analysis.setComputeMoments(True)

For the next step, we need to create an `AnalyzeTool` that, given a model file and a motion file, to perform the requested analysis. We have to create an `AnalyzeTool` for the normal gait and another one for the crouch gait. This may take a few seconds.

In [ ]:
## Normal Gait.

# Create an AnalyzeTool for normal gait.
analyze_tool_normal_gait = osim.AnalyzeTool()
analyze_tool_normal_gait.setName("Muscle_Analysis_Normal_Gait")

# Set the model file and motion file to analyze.
analyze_tool_normal_gait.setModelFilename("gait2392.osim")
analyze_tool_normal_gait.setCoordinatesFileName("normal_gait.mot")

# Add the MuscleAnalysis to the AnalyzeTool.
analyze_tool_normal_gait.updAnalysisSet().cloneAndAppend(muscle_analysis)

# Directory where results are stored.
analyze_tool_normal_gait.setResultsDir("MA_Normal_Gait_Results")

# Configure AnalyzeTool.
analyze_tool_normal_gait.setReplaceForceSet(False)
analyze_tool_normal_gait.setSolveForEquilibrium(True)
analyze_tool_normal_gait.setStartTime(first_time)
analyze_tool_normal_gait.setFinalTime(last_time)

# Print configuration of the AnalyzeTool to an XML file.
analyze_tool_normal_gait.printToXML("Muscle_Analysis_Normal_Gait_AnalyzeTool_setup.xml")

# Load configuration and run the analyses. 
analyze_tool_normal_gait = osim.AnalyzeTool("Muscle_Analysis_Normal_Gait_AnalyzeTool_setup.xml", True)
result_normal_gait = analyze_tool_normal_gait.run()


## Crouch Gait.

# Create an AnalyzeTool for crouch gait.
analyze_tool_crouch_gait = osim.AnalyzeTool()
analyze_tool_crouch_gait.setName("Muscle_Analysis_Crouch_Gait")

# Set the model file and motion file to analyze.
analyze_tool_crouch_gait.setModelFilename("gait2392.osim")
analyze_tool_crouch_gait.setCoordinatesFileName("crouch_gait.mot")

# Add the Muscleanalysis to the AnalyzeTool.
analyze_tool_crouch_gait.updAnalysisSet().cloneAndAppend(muscle_analysis)

# Directory where results are stored.
analyze_tool_crouch_gait.setResultsDir("MA_Crouch_Gait_Results")

# Configure AnalyzeTool.
analyze_tool_crouch_gait.setReplaceForceSet(False)
analyze_tool_crouch_gait.setSolveForEquilibrium(True)
analyze_tool_crouch_gait.setStartTime(first_time)
analyze_tool_crouch_gait.setFinalTime(last_time)

# Print configuration of the AnalyzeTool to an XML file.
analyze_tool_crouch_gait.printToXML("Muscle_Analysis_Crouch_Gait_AnalyzeTool_setup.xml")

# Load configuration and run the analyses. 
analyze_tool_crouch_gait = osim.AnalyzeTool("Muscle_Analysis_Crouch_Gait_AnalyzeTool_setup.xml", True)
result_crouch_gait = analyze_tool_crouch_gait.run()

Now that the analysis has been performed, you can extract the fiber length of the semitendinous muscle!

In [ ]:
## Normal gait

# Get results from file.
table_fiber_length_normal_gait = osim.TimeSeriesTable(
    "MA_Normal_Gait_Results/Muscle_Analysis_Normal_Gait_MuscleAnalysis_FiberLength.sto")

# Get values.
time_normal_gait = table_fiber_length_normal_gait.getIndependentColumn()
fiber_length_semitendinous_normal_gait = table_fiber_length_normal_gait.getDependentColumn("semiten_r")


## Crouch gait

# Get results from file.
table_fiber_length_crouch_gait = osim.TimeSeriesTable(
    "MA_Crouch_Gait_Results/Muscle_Analysis_Crouch_Gait_MuscleAnalysis_FiberLength.sto")

# Get values.
time_crouch_gait = table_fiber_length_crouch_gait.getIndependentColumn()
fiber_length_semitendinous_crouch_gait = table_fiber_length_crouch_gait.getDependentColumn("semiten_r")

Let's create a plot to analyze the fiber length of the semitendinous muscle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create six subplots, with 2 rows and 3 columns.
fig, axs = plt.subplots(1, 1, figsize=(7, 5))
fig.suptitle('Normal Gait vs. Crouch Gait')

# Plot the knee angles on the first subplot.
axs.plot(time_normal_gait, fiber_length_semitendinous_normal_gait.to_numpy(), label='normal_gait_semitendinous_fiber_length')
axs.plot(time_crouch_gait, fiber_length_semitendinous_crouch_gait.to_numpy(), label='crouch_gait_semitendinous_fiber_length')
axs.set_title('Fiber Length')
axs.set_xlabel('Time')
axs.set_ylabel('Fiber Length')
axs.grid()
axs.legend()

# Set the spacing between subplots
plt.subplots_adjust(wspace=0.5, hspace=0.5)

**Questions: Hamstrings Length**

**1.** Study the curves. Based on the plot, how do the peak hamstring lengths in normal and crouch gait compare? For this patient, would you recommend a hamstrings lengthening surgery? 

**2.** What are some limitations of your analysis?

## 4.6. Conclusion

In this tutorial, you explored a model of the lower extremity. Then, you have used this model to study lower limb kinematics using motion files from previous simulation results. Finally, you have performed a `MuscleAnalysis` to determine the fiber length of the semitendinous muscle to analyze possible underlying causes of crouch gait.

## 4.7. Useful Links


> **OpenSim Website:** https://opensim.stanford.edu/
>
> **OpenSim Python Scripting:** https://opensimconfluence.atlassian.net/wiki/spaces/OpenSim/pages/53085346/Scripting+in+Python
>
> **OpenSim API Documentation:** https://simtk.org/api_docs/opensim/api_docs/
> 
> **OpenSim Creator Website:** https://opensimcreator.com/
> 
> **SimTK Website:** https://simtk.org/projects/opensim
> 
> **Biomechanics of Movement Course Videos:** https://www.youtube.com/channel/UCDNGy0KKNLQ-ztcL5h2Z6zA

## 4.8 Acknowledgments

Allison Arnold and Scott Delp with help from many others.

Thanks to [OpenSimColab](https://simtk.org/projects/opencolab) project [[5]](https://doi.org/10.1080/10255842.2022.2104607) for creating the first OpenSim package for Google Colab and notebook-based workflows.

## 4.9. References

> [1]. Uchida, T. K., Delp, S. L., & Delp, D. (2021). **Biomechanics of movement: The science of sports, robotics, and rehabilitation.** *MIT Press*. https://mitpress.mit.edu/9780262044202/biomechanics-of-movement/
>
> [2]. Delp, S. L., Anderson, F. C., Arnold, A. S., Loan, P., Habib, A., John, C. T., Guendelman, E., & Thelen, D. G. (2007). **OpenSim: open-source software to create and analyze dynamic simulations of movement.** *IEEE Transactions on Bio-Medical Engineering*, 54(11), 1940–1950. https://doi.org/10.1109/TBME.2007.901024
>
> [3]. Delp, S. L., Loan, J. P., Hoy, M. G., Zajac, F. E., Topp, E. L., & Rosen, J. M. (1990). **An interactive graphics-based model of the lower extremity to study orthopaedic surgical procedures.** *IEEE Transactions on Bio-Medical Engineering*, 37(8), 757–767. https://doi.org/10.1109/10.102791
> 
> [4]. Arnold, A. S., Liu, M. Q., Schwartz, M. H., Ounpuu, S., & Delp, S. L. (2006). **The role of estimating muscle-tendon lengths and velocities of the hamstrings in the evaluation and treatment of crouch gait.** *Gait & Posture*, 23(3), 273–281. https://doi.org/10.1016/j.gaitpost.2005.03.003
>
> [5] Mokhtarzadeh, H., Jiang, F., Zhao, S., & Malekipour, F. (2022). **OpenColab project: OpenSim in Google colaboratory to explore biomechanics on the web.** *Computer Methods in Biomechanics and Biomedical Engineering*, 1–9. https://doi.org/10.1080/10255842.2022.2104607